# Parquet Optimization Demo

This notebook demonstrates how organizing data in Parquet files with appropriate row groups can significantly improve query performance. We'll compare:

1. **Default row grouping**: Standard Parquet file with default row group size
2. **File-based row grouping**: Row groups organized by source file for efficient filtering

In [1]:
import os
import time
from pathlib import Path

import duckdb
import pandas as pd
from dotenv import load_dotenv

from bettercode.taxi_utils import (
    download_taxi_data,
    get_data_dirs,
)

load_dotenv()
DATADIR = Path(os.getenv("DATADIR"))
orig_dir, preproc_dir = get_data_dirs(DATADIR)

In [2]:
download_taxi_data(DATADIR, delay_between_downloads=30)

Overall progress: 100%|██████████| 120/120 [00:00<00:00, 91462.20it/s]


In [3]:
## Step 1: Load all data files and combine with source filename

In [4]:
# Load all parquet files from orig_dir and add source filename column
import pyarrow as pa
import pyarrow.parquet as pq

data_files = list(orig_dir.glob("*.parquet"))
print(f"Found {len(data_files)} data files")

# Load each file and add the source filename
dfs = []
for file_path in sorted(data_files):
    df = pd.read_parquet(file_path)
    df['source_file'] = file_path.name  # Add filename as a column
    dfs.append(df)
    print(f"Loaded {file_path.name}: {len(df):,} rows")

# Combine all dataframes
combined_df = pd.concat(dfs, ignore_index=True)
print(f"\nTotal combined rows: {len(combined_df):,}")
print(f"Memory usage: {combined_df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
combined_df.head()

Found 120 data files
Loaded yellow_tripdata_2015-01.parquet: 12,741,035 rows
Loaded yellow_tripdata_2015-02.parquet: 12,442,394 rows
Loaded yellow_tripdata_2015-03.parquet: 13,342,951 rows
Loaded yellow_tripdata_2015-04.parquet: 13,063,758 rows
Loaded yellow_tripdata_2015-05.parquet: 13,157,677 rows
Loaded yellow_tripdata_2015-06.parquet: 12,324,936 rows
Loaded yellow_tripdata_2015-07.parquet: 11,559,666 rows
Loaded yellow_tripdata_2015-08.parquet: 11,123,123 rows
Loaded yellow_tripdata_2015-09.parquet: 11,218,122 rows
Loaded yellow_tripdata_2015-10.parquet: 12,307,333 rows
Loaded yellow_tripdata_2015-11.parquet: 11,305,240 rows
Loaded yellow_tripdata_2015-12.parquet: 11,452,996 rows
Loaded yellow_tripdata_2016-01.parquet: 10,905,067 rows
Loaded yellow_tripdata_2016-02.parquet: 11,375,412 rows
Loaded yellow_tripdata_2016-03.parquet: 12,203,824 rows
Loaded yellow_tripdata_2016-04.parquet: 11,927,996 rows
Loaded yellow_tripdata_2016-05.parquet: 11,832,049 rows
Loaded yellow_tripdata_2016

KeyboardInterrupt: 

## Step 2: Save with default row group size

In [ ]:
# Save with default row group size
default_file = preproc_dir / "combined_default.parquet"
combined_df.to_parquet(default_file, engine='pyarrow', index=False)

# Check file metadata
parquet_file = pq.ParquetFile(default_file)
print(f"File size: {default_file.stat().st_size / 1024**2:.2f} MB")
print(f"Number of row groups: {parquet_file.num_row_groups}")
print(f"Rows per group (avg): {len(combined_df) / parquet_file.num_row_groups:.0f}")

## Step 3: Save with row groups organized by source file

This approach creates one row group per source file, which allows Parquet to skip entire row groups when filtering by filename.

In [ ]:
# Sort by source file to cluster related rows together
df_sorted = combined_df.sort_values("source_file")

# Save with one row group per source file
optimized_file = preproc_dir / "combined_optimized.parquet"

# Get the schema from the dataframe
table = pa.Table.from_pandas(df_sorted)
schema = table.schema

# Write with one row group per source file
with pq.ParquetWriter(optimized_file, schema) as writer:
    for source_file, group_df in df_sorted.groupby("source_file", sort=False):
        table = pa.Table.from_pandas(group_df, schema=schema)
        writer.write_table(table)
        print(f"Wrote row group for {source_file}: {len(group_df):,} rows")

# Check file metadata
parquet_file = pq.ParquetFile(optimized_file)
print(f"\nFile size: {optimized_file.stat().st_size / 1024**2:.2f} MB")
print(f"Number of row groups: {parquet_file.num_row_groups}")
print(f"Rows per group (avg): {len(df_sorted) / parquet_file.num_row_groups:.0f}")

## Step 4: Benchmark loading and querying performance

We'll test:
1. **Full load time**: How long it takes to load the entire file
2. **Filtered query time**: How long it takes to load data for a specific source file

The optimized file should show significant improvement for filtered queries because it can skip row groups that don't match the filter.

In [ ]:
# Pick a specific file to query
test_filename = sorted(combined_df['source_file'].unique())[len(data_files)//2]  # Pick middle file
print(f"Testing queries for file: {test_filename}")
print(f"Expected rows: {len(combined_df[combined_df['source_file'] == test_filename]):,}")

### Benchmark 1: Full load (no filtering)

In [ ]:
# Test full load - default file
n_runs = 5
times_default_full = []
for i in range(n_runs):
    start = time.time()
    df_test = pd.read_parquet(default_file)
    elapsed = time.time() - start
    times_default_full.append(elapsed)
    
avg_default_full = sum(times_default_full) / len(times_default_full)
print(f"Default file - Full load: {avg_default_full:.4f}s (avg of {n_runs} runs)")
print(f"  Rows loaded: {len(df_test):,}")

# Test full load - optimized file
times_optimized_full = []
for i in range(n_runs):
    start = time.time()
    df_test = pd.read_parquet(optimized_file)
    elapsed = time.time() - start
    times_optimized_full.append(elapsed)
    
avg_optimized_full = sum(times_optimized_full) / len(times_optimized_full)
print(f"Optimized file - Full load: {avg_optimized_full:.4f}s (avg of {n_runs} runs)")
print(f"  Rows loaded: {len(df_test):,}")

print(f"\nSpeedup: {avg_default_full/avg_optimized_full:.2f}x")

### Benchmark 2: Filtered query (single source file)

This is where the optimized file should really shine - it can skip entire row groups.

In [ ]:
# Test filtered query - default file
times_default_filtered = []
for i in range(n_runs):
    start = time.time()
    df_test = pd.read_parquet(
        default_file, 
        filters=[('source_file', '==', test_filename)]
    )
    elapsed = time.time() - start
    times_default_filtered.append(elapsed)
    
avg_default_filtered = sum(times_default_filtered) / len(times_default_filtered)
print(f"Default file - Filtered query: {avg_default_filtered:.4f}s (avg of {n_runs} runs)")
print(f"  Rows loaded: {len(df_test):,}")

# Test filtered query - optimized file
times_optimized_filtered = []
for i in range(n_runs):
    start = time.time()
    df_test = pd.read_parquet(
        optimized_file, 
        filters=[('source_file', '==', test_filename)]
    )
    elapsed = time.time() - start
    times_optimized_filtered.append(elapsed)
    
avg_optimized_filtered = sum(times_optimized_filtered) / len(times_optimized_filtered)
print(f"Optimized file - Filtered query: {avg_optimized_filtered:.4f}s (avg of {n_runs} runs)")
print(f"  Rows loaded: {len(df_test):,}")

speedup = avg_default_filtered / avg_optimized_filtered
print(f"\n🚀 Filtered query speedup: {speedup:.2f}x")
print(f"Time saved: {(avg_default_filtered - avg_optimized_filtered)*1000:.1f}ms per query")

### Summary: Performance Comparison

Visualize the performance difference between the two approaches.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Create comparison chart
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Full load comparison
categories = ['Default', 'Optimized']
times_full = [avg_default_full, avg_optimized_full]
colors = ['#ff6b6b', '#4ecdc4']

ax1.bar(categories, times_full, color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)
ax1.set_ylabel('Time (seconds)', fontsize=12)
ax1.set_title('Full Load Performance', fontsize=14, fontweight='bold')
ax1.set_ylim(0, max(times_full) * 1.2)
for i, v in enumerate(times_full):
    ax1.text(i, v + max(times_full)*0.02, f'{v:.4f}s', ha='center', fontweight='bold')

# Filtered query comparison
times_filtered = [avg_default_filtered, avg_optimized_filtered]

ax2.bar(categories, times_filtered, color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)
ax2.set_ylabel('Time (seconds)', fontsize=12)
ax2.set_title(f'Filtered Query Performance ({speedup:.2f}x speedup)', 
              fontsize=14, fontweight='bold', color='green')
ax2.set_ylim(0, max(times_filtered) * 1.2)
for i, v in enumerate(times_filtered):
    ax2.text(i, v + max(times_filtered)*0.02, f'{v:.4f}s', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

print("\n" + "="*60)
print("KEY FINDINGS:")
print("="*60)
print(f"✓ Full load: Similar performance ({avg_default_full/avg_optimized_full:.2f}x)")
print(f"✓ Filtered query: {speedup:.2f}x faster with optimized row groups")
print(f"✓ Row groups in default file: {pq.ParquetFile(default_file).num_row_groups}")
print(f"✓ Row groups in optimized file: {pq.ParquetFile(optimized_file).num_row_groups} (one per source file)")
print("="*60)

## Explanation: Why Does This Work?

**Row Group Organization Matters for Query Performance:**

1. **Default Parquet files** write data in large row groups (often 1M rows) without considering data organization. When you filter by `source_file`, Parquet must:
   - Read metadata for all row groups
   - Scan through each row group to find matching rows
   - Can't skip row groups even if they don't contain the target file

2. **Optimized Parquet files** organize row groups by the filter column (`source_file`). When filtering:
   - Parquet reads row group statistics (min/max values per column)
   - Immediately skips row groups that don't contain the target filename
   - Only reads the specific row group(s) containing matching data
   - **Result**: Dramatically faster queries when filtering by the organized column

**Best Practices:**
- Organize row groups by columns you frequently filter on
- Sort data before writing to maximize compression
- Consider partition columns for very large datasets
- Balance row group size: too small = overhead, too large = poor filtering